# Kaggle Autonomous Multi-Agent System (KAMAS)
## Competition: Playground Series Season 6 Episode 9 (`playground-series-s6e9`)
### Phase 8 (Domain Features) & Phase 9 (Diverse Model Exploration) & Phase 10 (Ensembling)

This notebook trains and blends 3 distinct model families with domain-engineered features on Kaggle:
1. **LightGBM** (Leaf-wise histogram GBDT)
2. **CatBoost** (Symmetric oblivious trees with native categorical handling & GPU acceleration)
3. **XGBoost** (Depth-wise histogram GBDT with CUDA acceleration)
4. **Phase 10 Blender** (Rank averaging and optimal Nelder-Mead ensembling)

> **Hardware Recommendation:** Turn on **GPU T4 x2** in the right panel under Notebook settings for maximum speed.

### 1. Synchronize Repository & Setup Environment
Pull the latest codebase containing the Phase 8 domain feature generator and Phase 9 models.

In [ ]:
import os
import sys
from pathlib import Path

REPO_NAME = "electric-vehicle"
repo_path = Path(f"/kaggle/working/{REPO_NAME}")

if repo_path.exists():
    %cd /kaggle/working/electric-vehicle
    !git pull origin main
else:
    %cd /kaggle/working
    !git clone https://github.com/tuboa2/electric-vehicle-purchase-predictor.git electric-vehicle
    %cd /kaggle/working/electric-vehicle

sys.path.insert(0, str(Path.cwd()))
print(f"[*] Ready in: {Path.cwd()}")

### 2. Verify GPU & Python Environment

In [ ]:
!nvidia-smi
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
import polars as pl

print(f"[+] LightGBM: {lgb.__version__}")
print(f"[+] CatBoost: {cb.__version__}")
print(f"[+] XGBoost:  {xgb.__version__}")
print(f"[+] Polars:   {pl.__version__}")

### 3. Model 1: LightGBM with Domain Features (Phase 8 Hypothesis 1)
Trains 5-fold LightGBM on the engineered interaction and behavioral gating features.

In [ ]:
!python scripts/kaggle_train.py --model lgbm --features domain

### 4. Model 2: CatBoost with GPU & Native Categoricals (Phase 9 Hypothesis 2)
Trains 5-fold CatBoost with symmetric oblivious trees, GPU acceleration, and automatic categorical combination exploration.

In [ ]:
!python scripts/kaggle_train.py --model catboost --features domain

### 5. Model 3: XGBoost with CUDA Acceleration (Phase 9 Model Diversity)
Trains 5-fold XGBoost utilizing histogram tree method (`tree_method='hist'`) on GPU.

In [ ]:
!python scripts/kaggle_train.py --model xgboost --features domain

### 6. Phase 10: Ensembling & Blending Engine
Scans all trained models in `/kaggle/working/models/`, evaluates candidate correlations, optimizes blend weights via Nelder-Mead, computes rank averaging, and emits the verified `/kaggle/working/submission.csv`.

In [ ]:
!python scripts/kaggle_blend.py

### 7. Inspect Final Submission & Leaderboard Readiness

In [ ]:
import pandas as pd
sub_file = Path("/kaggle/working/submission.csv")
if sub_file.exists():
    df = pd.read_csv(sub_file)
    print(f"[+] File size: {sub_file.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"[+] Row count: {len(df):,}")
    print(f"[+] Nulls:     {df.isnull().sum().to_dict()}")
    print("\nSample predictions:")
    print(df.head(10))
else:
    print("[!] submission.csv not found!")